In [ ]:
%matplotlib inline
%reload_ext autoreload
%autoreload 2

import colorcet
from IPython.display import clear_output, display, HTML
import ipywidgets
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import os
from pathlib import Path
import pickle
from rastermap import Rastermap
import scipy
import sklearn
import sys

import pytoolsAL as ptAL
import subprocess

sys.path.append('../src')

globs = ptAL.globs.Globs(machine='labpc')
dd = globs.datadir
plt.style.use(globs.stylesheet) # path to stylesheet file
repo_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
data_dir = repo_root / 'data'


In [ ]:
# load main experiment
# mn = 'AL_0041'; td = '2025-11-07'; en = '1'
# mn = 'AL_0041'; td = '2025-11-03'; en = '1'
# mn = 'AL_0041'; td = '2025-10-17'; en = '1'
# mn = 'AL_0041'; td = '2025-11-20'; en = '2'
# mn = 'AL_0041'; td = '2025-11-21'; en = '1'
mn = 'AL_0041'; td = '2026-02-17'; en = '1'

expdir = Path(r'Z:\Subjects') / mn / td / en

gx = np.squeeze(np.load(expdir / 'galvoXPositions_mm.npy'))
gy = np.squeeze(np.load(expdir / 'galvoYPositions_mm.npy'))

# rounding error fix
gx = np.round(gx*2, 0)/2
gy = np.round(gy*2, 0)/2

laserAmp = np.squeeze(np.load(expdir / 'laserPowers_mW.npy'))
laserOnTimes = np.squeeze(np.load(expdir / 'laserOnTimes.npy'))
laserAmp = np.round(laserAmp, 1)
laserOffTimes = np.squeeze(np.load(expdir / 'laserOffTimes.npy'))

# filter for grid experiment
ess = np.squeeze(np.load(expdir / 'expStartStopTimes.npy')).reshape(-1, 2)
gridExp = ess[0]
theseTr = np.argwhere((laserOnTimes >= gridExp[0]) & (laserOnTimes <= gridExp[1])).ravel()

gx = gx[theseTr]
gy = gy[theseTr]
laserAmp = laserAmp[theseTr]
laserOnTimes = laserOnTimes[theseTr]
# laserOffTimes = laserOffTimes[theseTr]

n_comps = 50
svdTemp = np.squeeze(np.load(expdir / 'corr/svdTemporalComponents_corr.npy'))
svdTime = np.squeeze(np.load(expdir / 'corr/svdTemporalComponents_corr.timestamps.npy'))
svdSpat = np.squeeze(np.load(expdir / 'blue/svdSpatialComponents.npy'))[:, :, :n_comps]
meanImage = np.squeeze(np.load(expdir / 'blue/meanImage.npy'))

t_to_svd = scipy.interpolate.interp1d(svdTime, svdTemp[:len(svdTime), :n_comps], axis=0)
# t_to_svd_dv = scipy.interpolate.interp1d(tl_to_p(svdTime), np.diff(svdTemp[:, :n_comps], prepend=0), axis=0)

sys.path.append(r'C:\Users\anna\Repositories\Pipelines')
from widefield.widefield_deconv import deconvolve

svdTempDeconv = np.array([deconvolve(i) for i in svdTemp.T]).T

In [ ]:
iti = laserOnTimes[1:] - laserOffTimes[:-1]
print(f'mean: {np.mean(iti)}', f's.d.: {np.std(iti)}')

In [ ]:
positions_all = np.vstack((gx,  gy)).T
positions_mm = np.unique(positions_all, axis=0)

# convert mm pos to ix
pos_ix = np.vstack((-1*positions_mm[:, 1]+3, positions_mm[:, 0]+3.5)).T
pos_ix = pos_ix.astype('int')

# bregma = [305, 245] # 10/17
# bregma = [290, 245] # 11/03
# bregma = [300, 250] # 11/07
# bregma = [300, 235] # 11/20
bregma = [325, 245] # 11/21
xscale = 57.8
yscale = -57.8 # invert to plot correctly

xy_pos_px = np.zeros(positions_mm.shape)
xy_pos_px[:, 0] = bregma[0]+(positions_mm[:, 0]*xscale)
xy_pos_px[:, 1] = bregma[1]+(positions_mm[:, 1]*yscale)

In [ ]:
pos_n = 37
pos_roix = [int(xy_pos_px[pos_n, 0]-15), int(xy_pos_px[pos_n, 0]+15)]
pos_roiy = [int(xy_pos_px[pos_n, 1]-15), int(xy_pos_px[pos_n, 1]+15)]
roiSpat = np.mean(svdSpat[pos_roiy[0]:pos_roiy[1], pos_roix[0]:pos_roix[1], :], axis=(0, 1))
roiFluo = svdTempDeconv[:-1, :50] @ roiSpat
roiFluo = scipy.stats.zscore(roiFluo)

In [ ]:
pos_diff = np.linalg.norm(positions_all - positions_mm[pos_n, :], axis=1)
cm = mpl.cm.ScalarMappable(cmap='plasma_r')
cm.set_clim(0, 6)

In [ ]:
def gridSliderViewer(xlim):
    f = plt.figure(figsize=(6, 1))
    colors = cm.to_rgba(pos_diff)
    plt.scatter(laserOnTimes, np.ones(len(laserOnTimes))*5, marker='|', c=colors)
    plt.plot(svdTime, roiFluo, lw=0.5)
    plt.xlim(xlim)
    plt.show()

w = ipywidgets.interactive(gridSliderViewer, xlim=ipywidgets.FloatRangeSlider(
    value=[laserOnTimes[0], laserOnTimes[-1]], min=svdTime[0], max=svdTime[-1], step=0.1, 
    description='Time (s):', layout={'width': '80%'}))
display(w)

In [ ]:
# sanity check plot for positions in px
plt.imshow(meanImage, cmap='gray')
plt.scatter(*bregma, ec='m', fc='none', s=25)
plt.scatter(xy_pos_px[:, 0], xy_pos_px[:, 1], c='r', s=15, lw=0, alpha=0.3)

In [ ]:
# set up dff movie for animation
fps = 70
# window_wf = np.arange(0, 0.4, 1/fps) - 4/fps # trial bins
window_wf = np.arange(-0.2, 0.4, 1/fps)
base_ix = np.argmin(np.abs(window_wf - 0))
dff_ims = []

amp = 2.5
for iP, pos in enumerate(positions_mm):
    pos_x, pos_y = pos
    match_x = np.squeeze(np.argwhere(positions_all[:, 0] == pos_x))
    match_y = np.squeeze(np.argwhere(positions_all[:, 1] == pos_y))
    match_pos = np.intersect1d(match_x, match_y)
    match_amp = np.squeeze(np.argwhere(laserAmp == amp))
    match = np.intersect1d(match_pos, match_amp)
    these_times = laserOnTimes[match]

    wf_tr_times = [window_wf+i for i in these_times]

    tr_svd = t_to_svd(wf_tr_times)
    baseline = np.mean(tr_svd[:, :base_ix], axis=(0, 1))
    baseline_im = svdSpat @ baseline.T

    tr_mean = np.mean(tr_svd, axis=0)
    stim_ims = svdSpat @ tr_mean.T

    baseline_im += meanImage
    stim_ims = stim_ims.transpose(2, 0, 1) + baseline_im
    dffs = []
    for i in stim_ims:
        dff = (i - baseline_im) / baseline_im
        dffs.append(dff)
    dff_ims.append(np.array(dffs))
dff_ims = np.array(dff_ims)

In [ ]:
# animation for nick
ncol = 8
nrow = 8
f = plt.figure(figsize=(0.6*ncol, 0.6*nrow))
gs = mpl.gridspec.GridSpec(nrow, ncol, wspace=0.2, hspace=0.2)
clim_scale = 0.03
slow_x = 5

f.suptitle(f'{window_wf[0]*1000:.0f} ms after laser onset')
ims = []
# txt = plt.text(0.5, 0.93, 'LASER OFF', color='k', transform=f.transFigure, ha='center')
for iP, pos in enumerate(positions_mm):
    ax = plt.subplot(gs[*pos_ix[iP]])
    this_pos = dff_ims[iP]
    im = plt.imshow(this_pos[0], cmap='bwr', clim=np.r_[-1, 1]*clim_scale)
    plt.scatter(*xy_pos_px[iP], marker='o', ec='m', lw=0.5, fc='None', s=5)
    ax = ptAL.plotting.apply_image_defaults(ax)
    ims.append(im)

ax = plt.subplot(gs[:2, :2])
plt.imshow(meanImage, cmap='gray')
plt.scatter(xy_pos_px[:, 0], xy_pos_px[:, 1], marker='o', ec='m', lw=0.5, fc='None', s=5)
ax = ptAL.plotting.apply_image_defaults(ax)
ax = plt.subplot(gs[0, 5])
plt.imshow(np.zeros((560, 560)), cmap='bwr', clim=np.r_[-1, 1]*clim_scale*100)
ax = ptAL.plotting.apply_image_defaults(ax)
cb = ptAL.plotting.add_colorbar(ax, size='7%')
cb.set_label('% dF/F')
cb.set_ticks([clim_scale*-100, clim_scale*100])

# animate
def update(frame):
    # if window_wf[frame] > 0 and window_wf[frame] < 0.025:
    #     txt.set_text('LASER ON')
    #     txt.set_color('red')
    # else:
    #     txt.set_text('LASER OFF')
    #     txt.set_color('k')
    f.suptitle(f'{window_wf[frame]*1000:.0f} ms after laser onset ({slow_x}X slowed)')
    for i, im in enumerate(ims):
        im.set_array(dff_ims[i][frame])
    return ims

ani = mpl.animation.FuncAnimation(f, update, frames=len(window_wf), interval=1/fps, blit=True, repeat=True)
fn = f'gridMove_{mn}_{td}.mp4'
fpath = Path(r'C:\Users\anna\Data') / fn
ptAL.plotting.anim_to_file(ani, str(fpath), fps=int(35/slow_x), rewrite=True)

In [ ]:
# plot of grid experiment demo
window_wf = np.arange(0, 0.4, 1/35) - 4/35 # trial bins

ncol = 8
nrow = 8
f = plt.figure(figsize=(0.6*ncol, 0.6*nrow))
gs = mpl.gridspec.GridSpec(nrow, ncol, wspace=0.1, hspace=0.1)
t_plot = 7
amp = 4

dff_ims = []
baseline_ims = []
stim_ims = []

clim = 0.02
# f.suptitle(f'{window_wf[t_plot]*1000:.0f} ms after laser onset')
for iP, pos in enumerate(positions_mm):
    ax = plt.subplot(gs[pos_ix[iP, 0], pos_ix[iP, 1]])
    plt.title(f'{pos}', fontsize='xx-small', pad=0)
    pos_x, pos_y = pos
    match_x = np.squeeze(np.argwhere(positions_all[:, 0] == pos_x))
    match_y = np.squeeze(np.argwhere(positions_all[:, 1] == pos_y))
    match_pos = np.intersect1d(match_x, match_y)
    match_amp = np.squeeze(np.argwhere(laserAmp == amp))
    match = np.intersect1d(match_pos, match_amp)
    these_times = laserOnTimes[match]

    wf_tr_times = [window_wf+i for i in these_times]

    tr_svd = t_to_svd(wf_tr_times)
    baseline = np.mean(tr_svd[:, :4], axis=(0, 1))
    stim = np.mean(tr_svd[:, t_plot-2:t_plot+2], axis=(0, 1))

    baseline_im = svdSpat @ baseline.T
    stim_im = svdSpat @ stim.T

    baseline_im += meanImage
    stim_im += meanImage
    dff = (stim_im-baseline_im) / baseline_im

    plt.imshow(dff, cmap='bwr', clim=np.r_[-1, 1]*clim)
    plt.scatter(*xy_pos_px[iP], marker='.', c='m', lw=0, s=5)
    ax = ptAL.plotting.apply_image_defaults(ax)
    stim_ims.append(stim_im)
    dff_ims.append(dff)
    baseline_ims.append(baseline_im)

# ax = plt.subplot(gs[0, 0])
# plt.imshow(meanImage, cmap='gray')
# ax = ptAL.plotting.apply_image_defaults(ax)
ax = plt.subplot(gs[0, 0])
plt.imshow(np.zeros((560, 560)), cmap='bwr', clim=np.r_[-1, 1]*clim*100)
ax = ptAL.plotting.apply_image_defaults(ax)
cb = ptAL.plotting.add_colorbar(ax, size='7%')
cb.set_label('% dF/F')
cb.set_ticks([-clim*100, clim*100])

In [ ]:
sys.path.append(r'C:\Users\anna\Repositories\Pipelines')
from widefield.widefield_deconv import deconvolve

svdTempDeconv = np.array([deconvolve(i) for i in svdTemp.T]).T

In [ ]:
svdTempDeconv = np.array([deconvolve(i) for i in svdTemp.T]).T

In [ ]:
t_to_svd_deconv = scipy.interpolate.interp1d(svdTime, svdTempDeconv[:len(svdTime), :n_comps], axis=0)

In [ ]:
# plot of grid experiment demo - timecourses
window_wf = np.arange(0, 1.5, 1/70) - 12/35 # trial bins
base_ix = np.argmin(np.abs(window_wf - 0))
ncol = 8
nrow = 8
f = plt.figure(figsize=(0.7*ncol, 0.7*nrow))
gs = mpl.gridspec.GridSpec(nrow, ncol, wspace=0.5, hspace=0.5)
roi_around = 10 # radius
colors = ['dodgerblue']
# f.suptitle(f'{window_wf[t_plot]*1000:.0f} ms after laser onset')
all_dffs = []
for iP, pos in enumerate(positions_mm):
    ax = plt.subplot(gs[pos_ix[iP, 0], pos_ix[iP, 1]])
    plt.title(f'{pos}', fontsize='xx-small', pad=0)
    pos_x, pos_y = pos
    match_x = np.squeeze(np.argwhere(positions_all[:, 0] == pos_x))
    match_y = np.squeeze(np.argwhere(positions_all[:, 1] == pos_y))
    match_pos = np.intersect1d(match_x, match_y)
    amp_dffs = []
    for iA, amp in enumerate(np.unique(laserAmp)):
        match_amp = np.squeeze(np.argwhere(laserAmp == amp))
        match = np.intersect1d(match_pos, match_amp)
        these_times = laserOnTimes[match]

        wf_tr_times = [window_wf+i for i in these_times]

        # tr_svd = t_to_svd_deconv(wf_tr_times)
        tr_svd = t_to_svd(wf_tr_times)
        xROI = [int(xy_pos_px[iP, 0]-roi_around), int(xy_pos_px[iP, 0]+roi_around)]
        yRoi = [int(xy_pos_px[iP, 1]-roi_around), int(xy_pos_px[iP, 1]+roi_around)]
        thisSpat = np.mean(svdSpat[yRoi[0]:yRoi[1], xROI[0]:xROI[1], :], axis=(0, 1))

        fluo = tr_svd @ thisSpat
        # fluo = scipy.stats.zscore(fluo, axis=1)
        fluo += meanImage[yRoi[0]:yRoi[1], xROI[0]:xROI[1]].mean()
        fluo_baseline = np.mean(fluo[:, :base_ix], axis=(0, 1))
        dff = (fluo - fluo_baseline) / fluo_baseline
        tr_mean = np.median(dff, axis=0)
        amp_dffs.append(dff)

        tr_sem = scipy.stats.sem(dff, axis=0)
        plt.plot(window_wf, tr_mean, c=colors[iA], lw=0.5)
        plt.fill_between(window_wf, tr_mean - tr_sem, tr_mean + tr_sem, lw=0, alpha=0.3, color=colors[iA])

    all_dffs.append(amp_dffs)
    plt.xlim(window_wf[0], window_wf[-1])
    # plt.ylim(-1, 1)
    plt.ylim(np.r_[-1, 1]*0.03)
    plt.axhline(0, ls='--', c='k', lw=0.5)
    plt.axvspan(0, 0.025, ymin=0.9, ymax=0.95, color='r', lw=0, alpha=0.7)
    ax = ptAL.plotting.detick_despine(ax)

    tbar_s = 0.2
    zbar = 0.01
    x_start = window_wf[0]
    y_start = plt.ylim()[0]
    plt.plot([x_start, x_start+tbar_s], [y_start, y_start], c='k', lw=2, clip_on=False)
    plt.plot([x_start, x_start], [y_start, y_start+zbar], c='k', lw=2, clip_on=False)

    if np.all(pos == [-0.5, 3]):
        plt.text(x_start, y_start, f'{tbar_s*1000:.0f} ms', ha='left', va='top', fontsize='x-small')
        plt.text(x_start-0.02, y_start, f'{zbar*100:.0f} % dF/F', rotation='vertical', ha='right', va='bottom', fontsize='x-small')

all_dffs = np.array(all_dffs)
gridExp = {}
for iA, amp in enumerate(np.unique(laserAmp)):
    gridExp[np.unique(laserAmp)[iA]] = all_dffs[:, iA]
fname = f'gridExp_{mn}_{td}.npy'
np.save(Path(r'C:\Users\anna\Data\AL_0041\grid') / fname, gridExp)

In [ ]:
a = np.hstack(all_dffs)
a.shape

In [ ]:
# plot multiple sessions together
plotAmp = 4
all_dffs = []
for root, dirs, files in os.walk(r'C:\Users\anna\Data\AL_0041\grid'):
    for file in files:
        if file.endswith('.npy'):
            gridExp = np.load(Path(root) / file, allow_pickle=True).item()
            if plotAmp in gridExp.keys():
                all_dffs.append(gridExp[plotAmp])

all_dffs = np.hstack(all_dffs)

In [ ]:
# plot of grid experiment demo - timecourses
window_wf = np.arange(0, 1.5, 1/70) - 12/35 # trial bins
base_ix = np.argmin(np.abs(window_wf - 0))
ncol = 8
nrow = 8
f = plt.figure(figsize=(0.7*ncol, 0.7*nrow))
gs = mpl.gridspec.GridSpec(nrow, ncol, wspace=0.5, hspace=0.5)
roi_around = 10 # radius
colors = ['dodgerblue']

taus = []
for iP, pos in enumerate(positions_mm):
    ax = plt.subplot(gs[pos_ix[iP, 0], pos_ix[iP, 1]])
    dff = all_dffs[iP]
    tr_mean = np.median(dff, axis=0)
    amp_dffs.append(dff)
    
    # do exp rise fit
    def exp_rise(t, a, tau, offset):
        return a * (1 - np.exp(-t / tau)) + offset
    
    start_ix = 32
    popt, pcov = scipy.optimize.curve_fit(exp_rise, window_wf[start_ix:], tr_mean[start_ix:], 
                                          p0=[0.02, 0.1, 0], bounds=([0, 0, -np.inf], [0.1, 0.5, np.inf]))
    fit_y = exp_rise(window_wf[start_ix:], *popt)
    A_fit, tau_fit, offset_fit = popt
    taus.append(tau_fit)
    plt.plot(window_wf[start_ix:], fit_y, c='orange', lw=0.7, ls='--', zorder=20)

    tr_sem = scipy.stats.sem(dff, axis=0)
    plt.plot(window_wf, tr_mean, c=colors[iA], lw=0.5)
    plt.fill_between(window_wf, tr_mean - tr_sem, tr_mean + tr_sem, lw=0, alpha=0.3, color=colors[iA])

    plt.xlim(window_wf[0], window_wf[-1])
    # plt.ylim(-1, 1)
    plt.ylim(np.r_[-1, 1]*0.03)
    plt.axhline(0, ls='--', c='k', lw=0.5)
    plt.axvspan(0, 0.025, ymin=0.9, ymax=0.95, color='r', lw=0, alpha=0.7)
    ax = ptAL.plotting.detick_despine(ax)

    tbar_s = 0.2
    zbar = 0.01
    x_start = window_wf[0]
    y_start = plt.ylim()[0]
    plt.plot([x_start, x_start+tbar_s], [y_start, y_start], c='k', lw=2, clip_on=False)
    plt.plot([x_start, x_start], [y_start, y_start+zbar], c='k', lw=2, clip_on=False)

    if np.all(pos == [-0.5, 3]):
        plt.text(x_start, y_start, f'{tbar_s*1000:.0f} ms', ha='left', va='top', fontsize='x-small')
        plt.text(x_start-0.02, y_start, f'{zbar*100:.0f} % dF/F', rotation='vertical', ha='right', va='bottom', fontsize='x-small')

In [ ]:
f = plt.figure(figsize=(1.5, 1.5))
ax = plt.gca()
ax.set_aspect('equal')
ax = ptAL.plotting.detick_despine(ax)

coords = scipy.io.loadmat(data_dir / 'ctxOutlines.mat')['coords']
coords_transform = coords.copy()
for j in coords_transform.T:
    x = np.squeeze(j['x'][0])
    y = np.squeeze(j['y'][0])

    # convert from 10 um voxels and set bregma to [0, 0]
    cx = (x/100)-5.7
    cy = (y/-100)+5.2
    plt.plot(cx, cy, lw=0.5, c='lightgray', alpha=1, zorder=0)
cm = mpl.cm.ScalarMappable(cmap='cet_CET_L4')
clim = (0, 0.1)
cm.set_clim(clim)
plt.scatter(*positions_mm.T, fc=cm.to_rgba(taus), ec='k', s=30, clip_on=False)
cb = ptAL.plotting.add_colorbar(ax, mappable=cm, pad=0.06, size='6%')
cb.set_label('$\\tau$ (s)')
cb = ptAL.plotting.apply_colorbar_ticklabels(cb, clim)

In [ ]:
# plot of grid experiment demo - timecourses
window_wf = np.arange(0, 1.5, 1/70) - 12/35 # trial bins
base_ix = np.argmin(np.abs(window_wf - 0))
ncol = 8
nrow = 8
f = plt.figure(figsize=(0.7*ncol, 0.7*nrow))
gs = mpl.gridspec.GridSpec(nrow, ncol, wspace=0.2, hspace=0.2)
roi_around = 10 # radius
colors = ['powderblue', 'dodgerblue']

for iP, pos in enumerate(positions_mm):
    ax = plt.subplot(gs[pos_ix[iP, 0], pos_ix[iP, 1]])
    dff = all_dffs[iP]
    plt.imshow(dff, cmap='bwr', clim=np.r_[-1, 1]*0.08, aspect='auto', 
        extent=[window_wf[0], window_wf[-1], 0, dff.shape[0]])
    plt.axvspan(0, 0.025, ymin=1.01, ymax=1.06, color='r', lw=0, alpha=0.7, clip_on=False)
    ax = ptAL.plotting.detick_despine(ax)

In [ ]:
if plotAmp in gridExp.keys():
    print('asdf')